In [1]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data import SliceConditionDataset
from models import CustomVAE, DDPM, SliceConditionedTimeUNet

DATA_DIR = ROOT / "data" / "images"
VAE_CKPT = ROOT / "microlad-anode" / "vae_anode.pth"
PATCH_SIZE = 64
BATCH_SIZE = 2
TIMESTEPS = 1000
AXIS = "z"
SLICE_INDEX = 12
SEED = 0


In [2]:
dataset = SliceConditionDataset(
    DATA_DIR,
    patch_size=PATCH_SIZE,
    axis=AXIS,
    slice_index=SLICE_INDEX,
    seed=SEED,
)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
sample = next(iter(loader))
target, condition = sample["target"], sample["condition"]
axis, slice_index = sample["axis"], sample["slice_index"]
target.shape, condition.shape, axis, slice_index


(torch.Size([2, 1, 64, 64]),
 torch.Size([2, 1, 64, 64]),
 tensor([2, 2]),
 tensor([12, 12]))

In [3]:
vae = CustomVAE(latent_ch=4)
vae.load_state_dict(torch.load(VAE_CKPT, map_location="cpu")["vae"])
vae.eval()

unet = SliceConditionedTimeUNet(latent_ch=4, max_slices=64)
unet.eval()

ddpm = DDPM(timesteps=TIMESTEPS)


In [4]:
with torch.no_grad():
    target_z, _ = vae.encode(target * 2 - 1)
    condition_z, _ = vae.encode(condition * 2 - 1)
    t = torch.randint(0, TIMESTEPS, (target_z.shape[0],))
    noise = torch.randn_like(target_z)
    z_t = ddpm.q_sample(target_z, t, noise)
    pred_noise = unet(z_t, t, condition_z, axis, slice_index)

target_z.shape, condition_z.shape, z_t.shape, pred_noise.shape


(torch.Size([2, 4, 16, 16]),
 torch.Size([2, 4, 16, 16]),
 torch.Size([2, 4, 16, 16]),
 torch.Size([2, 4, 16, 16]))